2_MIMIC‑III

¿Quién lo creó y cómo se obtuvieron los datos?
MIMIC-III es una gran base de datos de un solo centro que comprende información relacionada con pacientes ingresados en unidades de cuidados intensivos de un gran hospital de atención terciaria. Fue desarrollado por el Laboratorio de Fisiología Computacional del MIT Institute for Medical Engineering and Science, en colaboración con el Beth Israel Deaconess Medical Center (BIDMC) de Boston, Massachusetts. Los autores principales son Alistair E.W. Johnson, Tom J. Pollard, Lu Shen, Li-Wei H. Lehman, Mengling Feng, Mohammad Ghassemi, Benjamin Moody, Peter Szolovits, Leo Anthony Celi y Roger G. Mark.
En cuanto a los sistemas de información clínica utilizados, durante el período de recolección de datos estuvieron activos dos sistemas distintos: el Philips CareVue Clinical Information System y el iMDsoft MetaVision ICU. Estos sistemas fueron la fuente de datos clínicos como mediciones fisiológicas verificadas por enfermería con marca de tiempo, como la documentación horaria de la frecuencia cardíaca, presión arterial y frecuencia respiratoria. Los datos fueron desidentificados de acuerdo con la normativa HIPAA para proteger la privacidad de los pacientes, y su acceso en la versión completa requiere un proceso de registro y certificación en PhysioNet.

¿De qué trata?
MIMIC-III es una base de datos grande y públicamente disponible que comprende datos de salud desidentificados de aproximadamente sesenta mil admisiones de pacientes que permanecieron en las unidades de cuidados intensivos del Beth Israel Deaconess Medical Center entre 2001 y 2012.

¿Qué contiene?
Los datos incluyen signos vitales, medicamentos, mediciones de laboratorio, observaciones y notas registradas por los proveedores de atención, balance de fluidos, códigos de procedimientos, códigos de diagnóstico, informes de imágenes, duración de la estadía hospitalaria, datos de supervivencia y más. MIMIC-III es una base de datos relacional compuesta por 26 tablas vinculadas mediante identificadores con el sufijo 'ID'. Por ejemplo, SUBJECT_ID hace referencia a un paciente único, HADM_ID a una admisión hospitalaria única, e ICUSTAY_ID a una admisión única en la UCI. La versión demo disponible libremente en PhysioNet contiene datos de 100 pacientes seleccionados de forma aleatoria del subconjunto de pacientes que fallecen, lo que significa que todos los pacientes tendrán fecha de defunción registrada.

Objetivo del modelo
Por sus características, MIMIC-III es ideal para múltiples tareas de modelado: predicción de mortalidad hospitalaria (clasificación binaria), predicción de la duración de la estancia en UCI (regresión), predicción de readmisión, o detección de sepsis. Con la versión demo, el objetivo más habitual es la predicción de mortalidad: dado el historial de un paciente en la UCI (signos vitales, resultados de laboratorio, medicamentos administrados), predecir si el paciente sobrevivirá o no. Dado que todos los pacientes del demo fallecieron, es importante considerarlo para el análisis de sesgos de muestreo.

In [1]:
# ============================================================
# LIBRERÍAS GENERALES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split


In [2]:
# ── PASO 1: CARGA DE LAS TABLAS PRINCIPALES ─────────────────
ruta = 'Datasets/2_MIMIC‑III/mimic-iii-clinical-database-demo-1.4/'

admissions = pd.read_csv(ruta + 'ADMISSIONS.csv')   # ingresos al hospital
patients   = pd.read_csv(ruta + 'PATIENTS.csv')     # datos del paciente
icustays   = pd.read_csv(ruta + 'ICUSTAYS.csv')     # estadías en UCI

print('ADMISSIONS:', admissions.shape)
print('PATIENTS:  ', patients.shape)
print('ICUSTAYS:  ', icustays.shape)

print('\nColumnas ADMISSIONS:', list(admissions.columns))
print('\nColumnas PATIENTS:  ', list(patients.columns))

ADMISSIONS: (129, 19)
PATIENTS:   (100, 8)
ICUSTAYS:   (136, 12)

Columnas ADMISSIONS: ['row_id', 'subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'admission_location', 'discharge_location', 'insurance', 'language', 'religion', 'marital_status', 'ethnicity', 'edregtime', 'edouttime', 'diagnosis', 'hospital_expire_flag', 'has_chartevents_data']

Columnas PATIENTS:   ['row_id', 'subject_id', 'gender', 'dob', 'dod', 'dod_hosp', 'dod_ssn', 'expire_flag']


In [3]:
# ── PASO 2: UNIR TABLAS POR subject_id ──────────────────────
# subject_id es la clave que identifica a cada paciente en todas las tablas
df_mimic = admissions.merge(
    patients[['subject_id', 'gender', 'dob', 'dod']],
    on='subject_id',
    how='left'
)

# Calcular EDAD al momento del ingreso
df_mimic['admittime'] = pd.to_datetime(df_mimic['admittime'])
df_mimic['dob']       = pd.to_datetime(df_mimic['dob'])
# Calculamos la edad usando solo el año para evitar el OverflowError
df_mimic['edad'] = df_mimic['admittime'].dt.year - df_mimic['dob'].dt.year
# Ajustamos el desfase de MIMIC (pacientes > 89 años aparecen con 300 años)
df_mimic.loc[df_mimic['edad'] > 100, 'edad'] = 90
df_mimic['edad'] = df_mimic['edad'].fillna(df_mimic['edad'].median())

# Calcular DURACIÓN DE LA ESTADÍA en días
df_mimic['dischtime'] = pd.to_datetime(df_mimic['dischtime'])
df_mimic['dias_estadia'] = (
    df_mimic['dischtime'] - df_mimic['admittime']
).dt.total_seconds() / 86400

print('Dataset unido:', df_mimic.shape)
print('\nVariable objetivo — hospital_expire_flag:')
print(df_mimic['hospital_expire_flag'].value_counts())
print('(0=sobrevivió, 1=murió)')

Dataset unido: (129, 24)

Variable objetivo — hospital_expire_flag:
hospital_expire_flag
0    89
1    40
Name: count, dtype: int64
(0=sobrevivió, 1=murió)


In [4]:
# ── PASO 3: CONVERTIR TEXTO A NÚMEROS ───────────────────────
# gender: M=0, F=1
df_mimic['gender_num'] = (df_mimic['gender'] == 'F').astype(float)

# admission_type: codificar con Categorical
df_mimic['tipo_ingreso'] = pd.Categorical(df_mimic['admission_type']).codes.astype(float)
df_mimic['seguro']       = pd.Categorical(df_mimic['insurance']).codes.astype(float)
df_mimic['estado_civil'] = pd.Categorical(df_mimic['marital_status'].fillna('Unknown')).codes.astype(float)

print('Tipos de ingreso:', df_mimic['admission_type'].unique())
print('Tipos de seguro: ', df_mimic['insurance'].unique())

Tipos de ingreso: ['EMERGENCY' 'ELECTIVE' 'URGENT']
Tipos de seguro:  ['Medicare' 'Private' 'Medicaid' 'Government']


In [5]:
# ── PASO 4: CONSTRUIR X e y ─────────────────────────────────
features_mimic = ['gender_num', 'edad', 'dias_estadia',
                  'tipo_ingreso', 'seguro', 'estado_civil']

X_raw = df_mimic[features_mimic].values.astype(float)
y_mimic = df_mimic['hospital_expire_flag'].values.astype(float)

# Imputar nulos que puedan quedar (rellenar con mediana de columna)
for j in range(X_raw.shape[1]):
    col = X_raw[:, j]
    nulos_idx = np.isnan(col)
    if nulos_idx.sum() > 0:
        col[nulos_idx] = np.nanmedian(col)

m_mimic = y_mimic.size
print('X shape:', X_raw.shape)
print('y shape:', y_mimic.shape)
print('Nulos en X:', np.isnan(X_raw).sum())

X shape: (129, 6)
y shape: (129,)
Nulos en X: 0


In [6]:
# ============================================================
# FUNCIÓN DE BALANCEO — oversampling con numpy
# ============================================================
def balancear(X, y):
    """
    Balancea un dataset desbalanceado usando OVERSAMPLING.
    
    ¿Qué hace?
    - Identifica cuántos ejemplos tiene cada clase
    - La clase con MÁS ejemplos queda igual
    - Las clases con MENOS ejemplos se repiten (con reemplazo)
      hasta tener la misma cantidad que la clase mayoritaria
    - Al final todas las clases tienen el mismo número de filas
    
    ¿Por qué oversampling y no undersampling?
    - Undersampling borra filas → perdemos información
    - Oversampling agrega filas → mantenemos toda la información original
    """
    clases = np.unique(y)
    n_max  = max(np.sum(y == c) for c in clases)   # tamaño de la clase más grande
    
    X_bal_list = []
    y_bal_list = []
    
    for c in clases:
        idx    = np.where(y == c)[0]               # índices de esta clase
        n_c    = len(idx)                           # cuántos ejemplos tiene
        
        if n_c < n_max:
            # repetir filas hasta alcanzar n_max
            extra  = n_max - n_c
            idx_extra = np.random.choice(idx, size=extra, replace=True)
            idx_final = np.concatenate([idx, idx_extra])
        else:
            idx_final = idx
        
        X_bal_list.append(X[idx_final])
        y_bal_list.append(y[idx_final])
    
    X_bal = np.concatenate(X_bal_list, axis=0)
    y_bal = np.concatenate(y_bal_list, axis=0)
    
    # Mezclar aleatoriamente para no dejar todas las clases juntas
    perm  = np.random.permutation(len(y_bal))
    return X_bal[perm], y_bal[perm]

def mostrar_balance(y, nombre, antes_despues='ANTES'):
    """Imprime cuántos ejemplos tiene cada clase."""
    clases, cuentas = np.unique(y, return_counts=True)
    print(f'  Balance {antes_despues} — {nombre}:')
    for c, n in zip(clases, cuentas):
        print(f'    Clase {int(c)}: {n} ({n/len(y)*100:.1f}%)')

np.random.seed(42)   # para reproducibilidad
print('Funciones de balanceo definidas')

Funciones de balanceo definidas


In [7]:
# ── BALANCEO ─────────────────────────────────
mostrar_balance(y_mimic, 'MIMIC-III', 'ANTES')

  Balance ANTES — MIMIC-III:
    Clase 0: 89 (69.0%)
    Clase 1: 40 (31.0%)


In [8]:
def featureNormalize(X):
    """
    Normaliza las features de X.
    Para cada columna: resta la media y divide por la desviación estándar.
    
    Retorna:
      X_norm : X normalizado (mismo tamaño que X)
      mu     : media de cada columna (se guarda para normalizar datos nuevos)
      sigma  : desviación estándar de cada columna
    """
    X_norm = X.copy()
    mu     = np.mean(X, axis=0)   # media de cada columna
    sigma  = np.std(X, axis=0)    # desviación estándar de cada columna
    X_norm = (X - mu) / sigma     # estandarización Z-score
    return X_norm, mu, sigma

In [10]:
# ── PASO 5: NORMALIZAR y AGREGAR COLUMNA DE UNOS ────────────
X_norm_m, mu_m, sigma_m = featureNormalize(X_raw)

#balancear
X_bal, y_bal = balancear(X_norm_m, y_mimic)

mostrar_balance(y_bal, 'MIMIC-III', 'DESPUÉS')

# Agregar columna de unos
m_mimic = len(y_bal)
X_mimic = np.concatenate([np.ones((m_mimic, 1)), X_bal], axis=1)
y_mimic = y_bal

print(f'\nX_mimic: {X_mimic.shape} | y_mimic: {y_mimic.shape}')


  Balance DESPUÉS — MIMIC-III:
    Clase 0: 89 (50.0%)
    Clase 1: 89 (50.0%)

X_mimic: (178, 7) | y_mimic: (178,)
